# OmniGuard Benchmark Plots
Run this from the repo root (`/Users/liushengwei/project/PythonProject/Project-OmniGuard`).
Requires: `pip install matplotlib seaborn scikit-learn`

In [3]:
import json, numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker
from collections import Counter
import os

# Auto-detect repo root
repo = os.path.abspath('.')
if not os.path.exists(os.path.join(repo, 'src')):
    # try notebook location
    repo = os.path.dirname(os.path.abspath('__file__'))

def load(path):
    with open(os.path.join(repo, path)) as f:
        return json.load(f)

pipe_raw = load('src/cloud-orchestrator/embodied_brain/benchmark_results/results.json')['results']
bl_raw   = load('src/cloud-orchestrator/embodied_brain/benchmark_results/baseline_results.json')['results']
ns_raw   = load('docs/experiment_results/benchmark_200_20260716_v2/no_safety_results.json')['results']

# Normalize field names
for r in pipe_raw:
    r['total_latency'] = r['router_latency'] + r['safety_latency'] + r['compiler_latency']
for r in ns_raw:
    r['total_latency'] = r['router_latency_ms'] + r['compiler_latency_ms']
    r['safety_latency'] = 0  # no safety gate in this ablation
for r in bl_raw:
    r['total_latency'] = r['latency_ms']

# PASS-only subsets
pipeline_pass = [r for r in pipe_raw if r['gate_decision'] == 'PASS']
ns_pass       = [r for r in ns_raw   if r['gate_decision'] == 'PASS']

COLORS = {'pipeline': '#1f77b4', 'baseline': '#ff7f0e', 'no_safety': '#2ca02c'}
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

ModuleNotFoundError: No module named 'matplotlib'

## Figure 1: Latency Distribution (PASS scenarios only)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 18000, 60)
for data, color, label in [
    ([r['total_latency'] for r in pipeline_pass], COLORS['pipeline'],
     f'Pipeline (n={len(pipeline_pass)}, {np.mean([r["total_latency"] for r in pipeline_pass]):.0f} ms)'),
    ([r['total_latency'] for r in ns_pass], COLORS['no_safety'],
     f'No Safety (n={len(ns_pass)}, {np.mean([r["total_latency"] for r in ns_pass]):.0f} ms)'),
    ([r['total_latency'] for r in bl_raw], COLORS['baseline'],
     f'Baseline (n={len(bl_raw)}, {np.mean([r["total_latency"] for r in bl_raw]):.0f} ms)'),
]:
    ax.hist(data, bins=bins, alpha=0.5, label=label, color=color, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('End-to-End Decision Latency (PASS scenarios)')
ax.legend()
plt.tight_layout()
plt.show()

## Figure 2: Per-Stage Latency Box Plot (Pipeline PASS)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
stage_data = {
    'Router':   [r['router_latency'] for r in pipeline_pass],
    'Safety':   [r['safety_latency'] for r in pipeline_pass],
    'Compiler': [r['compiler_latency'] for r in pipeline_pass],
}
bp = ax.boxplot(stage_data.values(), labels=stage_data.keys(), patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
for i, (k, v) in enumerate(stage_data.items()):
    ax.text(i+1, ax.get_ylim()[1]*0.95, f'\u03bc={np.mean(v):.0f}ms', ha='center', fontsize=10, color='gray')
ax.set_ylabel('Latency (ms)')
ax.set_title('Per-Agent Latency (Pipeline PASS)')
plt.tight_layout()
plt.show()

## Figure 3: Confusion Matrix Heatmap (Router Agent)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

pipe_router = [r for r in pipe_raw if r.get('router_label')]
true = [r['expected_label'] for r in pipe_router]
pred = [r['router_label'] for r in pipe_router]
labels = ['CRITICAL_OBSTACLE', 'NORMAL_NAV', 'SENSOR_ERROR']
cm = confusion_matrix(true, pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax,
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Router Confusion Matrix (n={len(pipe_router)}, accuracy={162/172*100:.2f}%)')
plt.tight_layout()
plt.show()

## Figure 4: Skip Savings by Gate

In [ ]:
# Determine which gate caused each skip
saved = {'PHYSICAL_HALT': [], 'ROUTER_SKIP': [], 'SAFETY_SKIP': []}
for r in pipe_raw:
    if r['gate_decision'] == 'HALT':
        saved['PHYSICAL_HALT'].append(1781 + 1202 + 3191)  # saved all 3 agents
    elif r['gate_decision'] == 'SKIP':
        if r.get('safety_decision') is not None:
            saved['SAFETY_SKIP'].append(r.get('compiler_latency', 3191) or 3191)
        else:
            saved['ROUTER_SKIP'].append(1202 + 3191)  # saved Safety + Compiler

fig, ax = plt.subplots(figsize=(8, 5))
names = list(saved.keys())
means = [np.mean(v)/1000 if v else 0 for v in saved.values()]
counts = [len(v) for v in saved.values()]
bars = ax.bar(names, means, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.75, width=0.5)
for bar, c, m in zip(bars, counts, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'n={c}\n{m:.1f}s', ha='center', fontsize=10)
ax.set_ylabel('Avg Latency Saved (s)')
ax.set_title(f'Circuit-Breaking Savings by Gate (n={sum(counts)} total skips)')
plt.tight_layout()
plt.show()

## Figure 5: Latency Comparison (All Configurations)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
configs = ['Baseline\n(Single Agent)', 'No Safety\n(Router\u2192Compiler)', 'Full Pipeline\n(Router\u2192Safety\u2192Compiler)']
lats   = [[r['total_latency'] for r in bl_raw],
          [r['total_latency'] for r in ns_pass],
          [r['total_latency'] for r in pipeline_pass]]
means  = [np.mean(x) for x in lats]
stds   = [np.std(x)  for x in lats]
bars = ax.bar(configs, [m/1000 for m in means], yerr=[s/1000 for s in stds], capsize=8,
              color=[COLORS['baseline'], COLORS['no_safety'], COLORS['pipeline']], alpha=0.75, width=0.5)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{m:.0f} ms\n\u00b1{s:.0f} ms', ha='center', fontsize=10)
ax.set_ylabel('Avg Latency (s)')
ax.set_title('End-to-End Latency: Baseline vs Ablations (PASS)')
plt.tight_layout()
plt.show()

## Figure 6: Critical Obstacle Detection Rate

In [ ]:
# Pipeline: detected = Router correctly labeled as CRITICAL
pipe_crit = [r for r in pipe_raw if r['type'] == 'critical']
pipe_det = sum(1 for r in pipe_crit if r.get('router_label') == 'CRITICAL_OBSTACLE' or r['gate_decision'] in ('HALT','SKIP'))
pipe_missed = len(pipe_crit) - pipe_det
# (Router labeled 29 correctly + 2 Safety-caught misclassifications SKIPped = 31)

ns_crit = [r for r in ns_raw if r['type'] == 'critical']
ns_det = sum(1 for r in ns_crit if r.get('router_label') == 'CRITICAL_OBSTACLE')
ns_missed = len(ns_crit) - ns_det

bl_crit = [r for r in bl_raw if r['type'] == 'critical']
bl_det = sum(1 for r in bl_crit if 'emergency_stop' in r.get('raw_output','').lower())
bl_missed = len(bl_crit) - bl_det

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, name, det, miss, color in zip(
        axes, 
        [f'Baseline\n(Seed 42)', f'No Safety Gate\n(Seed 42)', f'Full Pipeline\n(Seed 42)'],
        [bl_det, ns_det, pipe_det],
        [bl_missed, ns_missed, pipe_missed],
        [COLORS['baseline'], COLORS['no_safety'], COLORS['pipeline']]):
    total = det + miss
    ax.bar(['Detected', 'Missed'], [det, miss], color=[color, '#d62728'], alpha=0.75, width=0.5)
    for bar, v in zip(ax.patches, [det, miss]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(v), ha='center', fontsize=11, fontweight='bold')
    ax.set_title(f'{name}\n{det}/{total} = {det/total*100:.1f}%', fontsize=11)
    ax.set_ylim(0, max(det, miss)*1.4)
    ax.set_ylabel('Count')
plt.suptitle('Critical Obstacle Detection Rate', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Raw Numbers (copy-paste for thesis)

In [ ]:
def stats(vals):
    return f'n={len(vals)}, mean={np.mean(vals):.0f}, median={np.median(vals):.0f}, std={np.std(vals):.0f}, min={min(vals):.0f}, max={max(vals):.0f}'

print('=== PIPELINE PASS ===')
print(stats([r['total_latency'] for r in pipeline_pass]))
print(f'  Router:   {np.mean([r["router_latency"] for r in pipeline_pass]):.0f}')
print(f'  Safety:   {np.mean([r["safety_latency"] for r in pipeline_pass]):.0f}')
print(f'  Compiler: {np.mean([r["compiler_latency"] for r in pipeline_pass]):.0f}')

print('\n=== BASELINE (all) ===')
print(stats([r['total_latency'] for r in bl_raw]))

print('\n=== NO SAFETY PASS ===')
print(stats([r['total_latency'] for r in ns_pass]))
print(f'  Router:   {np.mean([r["router_latency_ms"] for r in ns_pass]):.0f}')
print(f'  Compiler: {np.mean([r["compiler_latency_ms"] for r in ns_pass]):.0f}')